In [4]:
# STAGE 3: ROBUST ENSEMBLE & FUSION
# ---------------------------------
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import os
import zipfile
from PIL import Image, ImageFile
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib
from google.colab import drive

# --- CRITICAL FIXES FOR NEW RUNTIME ---
# 1. Allow truncated images (prevents OSError crash)
ImageFile.LOAD_TRUNCATED_IMAGES = True

# 2. Mount Drive
drive.mount('/content/drive', force_remount=True)

# 3. Setup Paths (VERIFY THESE!)
PROJECT_FOLDER = 'Final Year Project'  # Change to 'Final Year Project' if you renamed it
PROJECT_PATH = os.path.join('/content/drive/My Drive', PROJECT_FOLDER)
DATA_PATH = os.path.join(PROJECT_PATH, 'Processed_Data', 'full_dataset_metadata.csv')
MODEL_PATH = os.path.join(PROJECT_PATH, 'Models')
ZIP_PATH = os.path.join(PROJECT_PATH, 'dataset.zip')
EXTRACT_ROOT = '/content/dataset_extraction'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"📂 Project Path: {PROJECT_PATH}")
print(f"⚙️ Using Device: {DEVICE}")

# 4. Smart Dataset Re-Extraction (Required for New Notebooks)
if not os.path.exists(EXTRACT_ROOT):
    print(f"⚠️ Dataset missing in this runtime. Extracting...")
    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(EXTRACT_ROOT)
        print("✅ Extraction complete!")
    else:
        raise FileNotFoundError(f"❌ Zip file not found at {ZIP_PATH}. Check your Drive folder name.")
else:
    print("✅ Dataset found locally.")

# --- FEATURE EXTRACTOR LOGIC ---

class UniversalFeatureExtractor(nn.Module):
    def __init__(self, model_name, path, num_classes):
        super().__init__()
        self.model_name = model_name

        print(f"Loading {model_name}...")

        # Architecture Definition
        if model_name == 'efficientnet':
            base = models.efficientnet_b5(weights=None)
            # Re-create classifier to load weights properly
            base.classifier[1] = nn.Linear(base.classifier[1].in_features, num_classes)
            base.load_state_dict(torch.load(path, map_location=DEVICE))
            # Replace classifier with Identity to get features
            base.classifier = nn.Identity()
            self.model = base

        elif model_name == 'densenet':
            base = models.densenet201(weights=None)
            base.classifier = nn.Linear(base.classifier.in_features, num_classes)
            base.load_state_dict(torch.load(path, map_location=DEVICE))
            # Replace classifier
            base.classifier = nn.Identity()
            self.model = base

        elif model_name == 'swin':
            base = models.swin_t(weights=None)
            base.head = nn.Linear(base.head.in_features, num_classes)
            base.load_state_dict(torch.load(path, map_location=DEVICE))
            # Replace head
            base.head = nn.Identity()
            self.model = base

        self.model.to(DEVICE).eval()

    def forward(self, x):
        return self.model(x)

# --- EXECUTION ---

# 1. Load Metadata
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Metadata CSV not found at {DATA_PATH}. Did Stage 1 finish?")

df = pd.read_csv(DATA_PATH)
classes = sorted(df['label'].unique())
num_classes = len(classes)
print(f"Detected {num_classes} classes: {classes}")

# 2. Initialize Extractors (Load all 3 models)
# We wrap them in try-except in case you skipped one in Stage 2
extractors = {}
try:
    extractors['effnet'] = UniversalFeatureExtractor('efficientnet', os.path.join(MODEL_PATH, 'efficientnet_best.pth'), num_classes)
    extractors['dense'] = UniversalFeatureExtractor('densenet', os.path.join(MODEL_PATH, 'densenet_best.pth'), num_classes)
    extractors['swin'] = UniversalFeatureExtractor('swin', os.path.join(MODEL_PATH, 'swin_best.pth'), num_classes)
except FileNotFoundError as e:
    print(f"⚠️ Warning: Could not load a model. Check Stage 2 output. Error: {e}")

# 3. Extraction & Fusion Function
def extract_features(split_name):
    print(f"\n--- Processing {split_name} Set ---")
    subset = df[df['split'] == split_name]

    # Standard Transform
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_feats = []
    env_feats = []
    labels = []

    total = len(subset)
    for i, (idx, row) in enumerate(subset.iterrows()):
        if i % 100 == 0: print(f"Processed {i}/{total} images...")

        # Load Image
        try:
            img_path = row['filepath']
            # Quick fix for path changes if you moved things
            if not os.path.exists(img_path):
                 # Try to find it in the new extraction root
                 img_path = img_path.replace('/content/dataset', '/content/dataset_extraction/dataset')

            img = Image.open(img_path).convert('RGB')
            img_t = transform(img).unsqueeze(0).to(DEVICE)

            # Extract Features from ALL models
            with torch.no_grad():
                f_eff = extractors['effnet'](img_t).cpu().numpy()
                f_dense = extractors['dense'](img_t).cpu().numpy()
                f_swin = extractors['swin'](img_t).cpu().numpy()

            # Fuse (Concatenate)
            combined = np.hstack([f_eff, f_dense, f_swin])

            # Env Features
            env = np.array([[row['temperature'], row['humidity'], row['ph']]])

            img_feats.append(combined)
            env_feats.append(env)
            labels.append(classes.index(row['label']))

        except Exception as e:
            print(f"Error processing {row['filepath']}: {e}")
            continue

    return np.vstack(img_feats), np.vstack(env_feats), np.array(labels)

# 4. Run Extraction
print("Extracting Training Features...")
X_img_train, X_env_train, y_train = extract_features('train')

print("Extracting Validation Features...")
X_img_val, X_env_val, y_val = extract_features('val')

# 5. Fusion & Meta-Learning
print("\n--- Training Fusion Model ---")
# Scale Environmental Data
scaler = StandardScaler()
X_env_train = scaler.fit_transform(X_env_train)
X_env_val = scaler.transform(X_env_val)

# Combine Image Embeddings + Env Data
X_train_final = np.hstack([X_img_train, X_env_train])
X_val_final = np.hstack([X_img_val, X_env_val])

print(f"Final Feature Vector Size: {X_train_final.shape[1]}")

# Meta Learner (Simple Neural Net)
meta_learner = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000, random_state=42)
meta_learner.fit(X_train_final, y_train)

# Evaluation
val_preds = meta_learner.predict(X_val_final)
acc = accuracy_score(y_val, val_preds)
print(f"\n🚀 Final Ensemble Accuracy: {acc*100:.2f}%")


Mounted at /content/drive
📂 Project Path: /content/drive/My Drive/Final Year Project
⚙️ Using Device: cuda
✅ Dataset found locally.
Detected 28 classes: ['Chilli Bacterial Spot', 'Chilli Cercospora Leaf Spot', 'Chilli Curl Virus', 'Chilli Healthy Leaf', 'Chilli Nutrition Deficiency', 'Chilli White spot', 'Cotton bacterial_blight', 'Cotton curl_virus', 'Cotton fussarium_wilt', 'Cotton healthy', 'Maize fall armyworm', 'Maize grasshoper', 'Maize healthy', 'Maize leaf beetle', 'Maize leaf blight', 'Maize leaf spot', 'Maize streak virus', 'Potato_Early_blight', 'Potato_Late_blight', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Potato_healthy', 'Tomato healthy', 'Tomato leaf blight', 'Tomato leaf curl', 'Tomato septoria leaf spot', 'Tomato verticulium wilt']
Loading efficientnet...
Loading densenet...
Loading swin...
Extracting Training Features...

--- Processing train Set ---
Processed 0/14227 images...
Processed 100/14227 images...
Processed 200/14227 images...
Pr

In [5]:
# FIX FOR REPORTING & SAVING
# --------------------------
from sklearn.metrics import classification_report


# Create a list of all label indices [0, 1, 2, ... 27]
all_labels = range(len(classes))

# We force the report to look for all labels, even if missing in y_val
print(classification_report(y_val, val_preds, labels=all_labels, target_names=classes))

# 6. Save Everything
print("Saving Fusion Models...")
joblib.dump(meta_learner, os.path.join(MODEL_PATH, 'meta_learner_final.pkl'))
joblib.dump(scaler, os.path.join(MODEL_PATH, 'env_scaler_final.pkl'))
print("✅ Stage 3 Complete! You are ready for the final demo.")

                             precision    recall  f1-score   support

      Chilli Bacterial Spot       1.00      1.00      1.00        43
Chilli Cercospora Leaf Spot       1.00      1.00      1.00        49
          Chilli Curl Virus       1.00      1.00      1.00       116
        Chilli Healthy Leaf       1.00      1.00      1.00       132
Chilli Nutrition Deficiency       1.00      1.00      1.00       126
          Chilli White spot       1.00      1.00      1.00        55
    Cotton bacterial_blight       1.00      1.00      1.00       125
          Cotton curl_virus       1.00      1.00      1.00       118
      Cotton fussarium_wilt       1.00      1.00      1.00       115
             Cotton healthy       1.00      1.00      1.00       113
        Maize fall armyworm       0.97      0.97      0.97        79
           Maize grasshoper       0.99      0.99      0.99       191
              Maize healthy       0.95      0.96      0.95        54
          Maize leaf beetle      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me